# Initial settings

In [ ]:
# ===== ユーザー設定 =====

DRIVE_PROJECT = "/content/drive/MyDrive/si_masterthesis_colab"

# Driveへアップロードした入力ZIP
INPUT_ZIP_NAME = "test_cube_seed1.zip"

# 結果の保存先
RESULT_FOLDER_NAME = "test"

# 解析条件
RHO = 0
SEED = 1
TEXTURE = "cube"
SD = 2
STATE = 13

# グラフのエッジ重み
WEIGHT = "boundary_length_over_distance"

# Connect to Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Check files

In [ ]:
from pathlib import Path

drive_project = Path(DRIVE_PROJECT)
input_zip = drive_project / "input_data" / INPUT_ZIP_NAME
result_dir = drive_project / "results" / RESULT_FOLDER_NAME

print("Driveプロジェクト:", drive_project)
print("入力ZIP:", input_zip)
print("入力ZIPの存在:", input_zip.exists())
print("結果保存先:", result_dir)

if not input_zip.exists():
    raise FileNotFoundError(
        f"入力ZIPが見つかりません:\n{input_zip}\n"
        "Google Drive上のフォルダ名とZIP名を確認してください。"
    )

result_dir.mkdir(parents=True, exist_ok=True)

# Take codes from GitHub

In [ ]:
import shutil
from pathlib import Path

repo_dir = Path("/content/pipeline")

# 再実行時に古い作業フォルダを削除
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone -q https://github.com/Sorao0921/biaxial-cpfem.git /content/pipeline

print("取得完了:", repo_dir)
print(
    "解析スクリプト:",
    (repo_dir / "tools/theme1/analyze_graph_spectra.py").exists()
)

# Install library

In [ ]:
%cd /content/pipeline

!pip install -q -e .

# Extract to Colab

In [ ]:
import shutil
import zipfile
from pathlib import Path

repo_dir = Path("/content/pipeline")
input_zip = Path(DRIVE_PROJECT) / "input_data" / INPUT_ZIP_NAME

with zipfile.ZipFile(input_zip, "r") as archive:
    archive.extractall(repo_dir)

print("展開完了:", repo_dir)

# Check files

In [ ]:
from pathlib import Path

repo_dir = Path("/content/pipeline")

spatial_dir = repo_dir / "database" / "spatial_model" / f"seed{SEED}"
outputs_dir = repo_dir / "outputs"

required_files = [
    spatial_dir / "elements.csv",
    spatial_dir / "nodes.csv",
]

print("=== 空間モデル ===")
for path in required_files:
    size_mb = path.stat().st_size / 1024**2 if path.exists() else 0
    print(f"{path}: exists={path.exists()}, size={size_mb:.1f} MB")

print("\n=== 対象ケース候補 ===")
pattern = f"*{TEXTURE}_sd{SD}_seed{SEED}*state{STATE:02d}*.csv"
matched = list(outputs_dir.rglob(pattern))

for path in matched:
    print(path.relative_to(repo_dir))

print(f"\n該当CSV数: {len(matched)}")

missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "空間モデルが不足しています:\n"
        + "\n".join(str(path) for path in missing)
    )

if len(matched) < 3:
    raise FileNotFoundError(
        "対象ケースのCSVが3種類揃っていない可能性があります。\n"
        "ZIP内部のoutputsフォルダ構成を確認してください。"
    )

# Case check

In [ ]:
%cd /content/pipeline

from src.dashboard.catalog import scan_outputs
from src.theme1.contribution import complete_cases
from src.config.pipeline_paths import OUTPUTS_DIR, SPATIAL_MODELS_DIR

records = scan_outputs(OUTPUTS_DIR, prefer_raw_height=True)
cases = complete_cases(records, SPATIAL_MODELS_DIR)

selected_cases = [
    case for case in cases
    if case.rho == RHO
    and case.seed == SEED
    and case.texture == TEXTURE
    and case.sd == SD
    and case.state == STATE
]

print("検出された全ケース数:", len(cases))
print("今回の対象ケース数:", len(selected_cases))

for case in selected_cases:
    print(case.case_id)

if not selected_cases:
    raise RuntimeError(
        "指定したケースを検出できませんでした。\n"
        "ファイル名、フォルダ構成、解析条件を確認してください。"
    )

# Run analysis

In [ ]:
%cd /content/pipeline

!python tools/theme1/analyze_graph_spectra.py \
    --outputs /content/pipeline/outputs \
    --spatial-models /content/pipeline/database/spatial_model \
    --output-dir /content/pipeline/database/theme1/graph_spectra \
    --rho {RHO} \
    --seed {SEED} \
    --texture {TEXTURE} \
    --sd {SD} \
    --state {STATE} \
    --weight {WEIGHT}

# Check result

In [ ]:
from pathlib import Path
import pandas as pd
import json

output_dir = Path("/content/pipeline/database/theme1/graph_spectra")

print("=== 生成ファイル ===")
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_dir))

diagnostics_path = output_dir / "diagnostics.json"

if diagnostics_path.exists():
    print("\n=== 診断情報 ===")
    print(json.loads(diagnostics_path.read_text(encoding="utf-8")))

energies_path = output_dir / "band_energies.csv"

if energies_path.exists():
    display(pd.read_csv(energies_path).head(20))

# Save to Drive

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

source_dir = Path("/content/pipeline/database/theme1/graph_spectra")
drive_results = Path(DRIVE_PROJECT) / "results" / RESULT_FOLDER_NAME

# 実行条件ごとに保存フォルダを分ける
run_name = (
    f"rho_{RHO}_{TEXTURE}_sd{SD}_"
    f"seed{SEED}_state{STATE:02d}"
)

destination = drive_results / run_name

if destination.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    destination = drive_results / f"{run_name}_{timestamp}"

shutil.copytree(source_dir, destination)

print("Driveへの保存完了:")
print(destination)